In [ ]:
import os, sys

HOME = "/home/j-j14d208"
CODE_DIR = f"{HOME}/2_ml_pipeline"
DATA_DIR = f"{HOME}/kospi200-project"

os.environ["BASE_PATH"] = DATA_DIR
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print(f"작업 디렉토리: {os.getcwd()}")

In [ ]:
!pip install matplotlib seaborn scikit-learn --quiet

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 11
print("라이브러리 로드 완료")

In [ ]:
# ========== 데이터 로드 ==========
BASE_PATH = Path(os.environ.get("BASE_PATH"))
df = pd.read_parquet(str(BASE_PATH / "processed" / "tft_features" / "tft_features.parquet"))
df["date"] = pd.to_datetime(df["date"])
df["target_5d"] = df["target_5d"].astype(int)

print(f"전체: {len(df):,} rows, {df['ticker'].nunique()} tickers")
print(f"기간: {df['date'].min().date()} ~ {df['date'].max().date()}")
print(f"타겟 분포:\n{df['target_5d'].value_counts()}")
print(f"\n타겟 비율: 상승 {df['target_5d'].mean()*100:.1f}% / 하락 {(1-df['target_5d'].mean())*100:.1f}%")
print(f"\n컬럼:\n{list(df.columns)}")

## 1. 피처-타겟 상관분석

각 피처가 target_5d(상승/하락)와 얼마나 상관이 있는지 분석합니다.
- **Point-biserial correlation**: 연속 피처 vs 이진 타겟 간 상관계수
- **KS test**: 상승/하락 그룹의 피처 분포가 통계적으로 다른지 검정
- 신호 강도: *** (|r|>0.03), ** (|r|>0.01), * (|r|>0.005), - (무신호)

In [ ]:
# ========== 피처-타겟 상관분석 ==========
observed_features = [
    "daily_return", "log_volume_change", "high_low_range",
    "rsi_14", "macd_norm", "bb_percent_b",
    "volume_ratio_5d", "momentum_5d", "momentum_20d",
    "foreign_net_ratio", "inst_net_ratio",
    "kospi200_return", "vix", "vix_change", "usd_krw_change",
    "relative_return",
]
observed_features = [c for c in observed_features if c in df.columns]

results = []
for feat in observed_features:
    col = df[feat].dropna()
    target = df.loc[col.index, "target_5d"]
    
    corr, p_val = stats.pointbiserialr(target, col)
    
    up = col[target == 1]
    down = col[target == 0]
    ks_stat, ks_p = stats.ks_2samp(up, down)
    
    results.append({
        "feature": feat,
        "corr": corr,
        "corr_p": p_val,
        "ks_stat": ks_stat,
        "ks_p": ks_p,
        "mean_up": up.mean(),
        "mean_down": down.mean(),
        "mean_diff": up.mean() - down.mean(),
        "signal": "***" if abs(corr) > 0.03 else ("**" if abs(corr) > 0.01 else ("*" if abs(corr) > 0.005 else "-")),
    })

corr_df = pd.DataFrame(results).sort_values("corr", key=abs, ascending=False)

print("=" * 90)
print("  피처-타겟 상관분석 (Point-biserial correlation)")
print("=" * 90)
print(corr_df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("=" * 90)

In [ ]:
# ========== 상관계수 시각화 ==========
fig, ax = plt.subplots(figsize=(12, 6))
colors = ["#e74c3c" if c < 0 else "#2ecc71" for c in corr_df["corr"]]
bars = ax.barh(range(len(corr_df)), corr_df["corr"].values, color=colors, alpha=0.8)
ax.set_yticks(range(len(corr_df)))
ax.set_yticklabels(corr_df["feature"].values)
ax.set_xlabel("Point-biserial Correlation with target_5d")
ax.set_title("Feature-Target Correlation")
ax.axvline(x=0, color="gray", linestyle="-", alpha=0.3)
ax.axvline(x=0.03, color="blue", linestyle="--", alpha=0.3, label="|r|=0.03")
ax.axvline(x=-0.03, color="blue", linestyle="--", alpha=0.3)
ax.legend()
ax.grid(axis="x", alpha=0.3)
for bar, val in zip(bars, corr_df["corr"].values):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2, f"{val:.4f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ========== Mutual Information (비선형 상관) ==========
from sklearn.feature_selection import mutual_info_classif

X = df[observed_features].fillna(0).values
y = df["target_5d"].values

mi_scores = mutual_info_classif(X, y, random_state=42, n_neighbors=5)
mi_df = pd.DataFrame({"feature": observed_features, "MI": mi_scores}).sort_values("MI", ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(range(len(mi_df)), mi_df["MI"].values, color="#3498db", alpha=0.8)
ax.set_yticks(range(len(mi_df)))
ax.set_yticklabels(mi_df["feature"].values)
ax.set_xlabel("Mutual Information Score")
ax.set_title("Feature-Target Mutual Information (비선형 상관 포함)")
ax.grid(axis="x", alpha=0.3)
for i, val in enumerate(mi_df["MI"].values):
    ax.text(val + 0.0001, i, f"{val:.4f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

print("\nMI 랭킹:")
print(mi_df.to_string(index=False))

## 2. 피처 간 상관관계 (다중공선성)

피처끼리 높은 상관이 있으면 중복 정보 — 모델에 노이즈가 됩니다.
- 상관계수 |r| > 0.7이면 다중공선성 의심
- 중복 피처는 하나를 제거하거나 PCA로 합칠 수 있음

In [ ]:
# ========== 피처 간 상관행렬 ==========
feat_corr = df[observed_features].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(feat_corr, dtype=bool))
sns.heatmap(feat_corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax, annot_kws={"size": 8})
ax.set_title("Feature-Feature Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.show()

# 높은 상관 피처 쌍
print("\n상관계수 |r| > 0.5 인 피처 쌍:")
high_corr = []
for i in range(len(observed_features)):
    for j in range(i+1, len(observed_features)):
        r = feat_corr.iloc[i, j]
        if abs(r) > 0.5:
            high_corr.append((observed_features[i], observed_features[j], r))
if high_corr:
    for f1, f2, r in sorted(high_corr, key=lambda x: abs(x[2]), reverse=True):
        print(f"  {f1:25s} <-> {f2:25s} : r = {r:+.3f}")
else:
    print("  없음")

## 3. 상승/하락 그룹별 피처 분포

분포가 겹칠수록 모델이 구분하기 어렵습니다. 상위 8개 피처의 분포를 비교합니다.

In [ ]:
# ========== 상승/하락 그룹별 분포 비교 ==========
top_features = corr_df["feature"].values[:8]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, feat in enumerate(top_features):
    ax = axes[idx]
    up_data = df[df["target_5d"] == 1][feat].dropna()
    down_data = df[df["target_5d"] == 0][feat].dropna()
    
    low, high = df[feat].quantile(0.01), df[feat].quantile(0.99)
    ax.hist(down_data.clip(low, high), bins=80, alpha=0.5, color="#e74c3c", label="하락", density=True)
    ax.hist(up_data.clip(low, high), bins=80, alpha=0.5, color="#2ecc71", label="상승", density=True)
    
    corr_val = corr_df[corr_df["feature"] == feat]["corr"].values[0]
    ax.set_title(f"{feat}\n(r={corr_val:.4f})", fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle("상승 vs 하락 피처 분포 비교 (Top 8)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. 시간에 따른 상관관계 안정성

피처-타겟 상관이 시간에 따라 변하면 특정 기간에만 유효한 피처일 수 있습니다.

In [ ]:
# ========== 시간별 상관관계 안정성 ==========
top6 = corr_df["feature"].values[:6]
df["year"] = df["date"].dt.year
yearly_corr = {}

for year in sorted(df["year"].unique()):
    year_df = df[df["year"] == year]
    if len(year_df) < 1000:
        continue
    corrs = {}
    for feat in top6:
        col = year_df[feat].dropna()
        target = year_df.loc[col.index, "target_5d"]
        if len(col) > 100:
            r, _ = stats.pointbiserialr(target, col)
            corrs[feat] = r
    yearly_corr[year] = corrs

yearly_df = pd.DataFrame(yearly_corr).T

fig, ax = plt.subplots(figsize=(14, 6))
for feat in top6:
    if feat in yearly_df.columns:
        ax.plot(yearly_df.index, yearly_df[feat], marker="o", linewidth=1.5, label=feat)

ax.axhline(y=0, color="gray", linestyle="-", alpha=0.3)
ax.set_xlabel("Year")
ax.set_ylabel("Point-biserial Correlation")
ax.set_title("Feature-Target Correlation Over Time")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("연도별 상관계수:")
print(yearly_df.round(4).to_string())

## 5. 종합 결론

In [ ]:
# ========== 종합 요약 ==========
print("=" * 70)
print("  피처-타겟 상관분석 종합 요약")
print("=" * 70)

strong = corr_df[corr_df["corr"].abs() > 0.03]
moderate = corr_df[(corr_df["corr"].abs() > 0.01) & (corr_df["corr"].abs() <= 0.03)]
weak = corr_df[(corr_df["corr"].abs() > 0.005) & (corr_df["corr"].abs() <= 0.01)]
none_sig = corr_df[corr_df["corr"].abs() <= 0.005]

print(f"\n[신호 강도 분류]")
print(f"  강한 신호 (|r|>0.03): {len(strong)}개 - {list(strong['feature'].values)}")
print(f"  중간 신호 (|r|>0.01): {len(moderate)}개 - {list(moderate['feature'].values)}")
print(f"  약한 신호 (|r|>0.005): {len(weak)}개 - {list(weak['feature'].values)}")
print(f"  무신호 (|r|<=0.005): {len(none_sig)}개 - {list(none_sig['feature'].values)}")

print(f"\n[다중공선성]")
if high_corr:
    print(f"  높은 상관 피처 쌍: {len(high_corr)}개")
    for f1, f2, r in high_corr[:5]:
        print(f"    {f1} <-> {f2}: r={r:+.3f}")
else:
    print("  높은 상관 피처 쌍 없음")

print(f"\n[권장 사항]")
if len(strong) > 0:
    print(f"  1. 강한 신호 피처 {len(strong)}개를 우선 사용")
if len(none_sig) > 0:
    print(f"  2. 무신호 피처 {len(none_sig)}개는 제거 고려")
if high_corr:
    print(f"  3. 높은 상관 피처 중 하나 제거 (다중공선성)")
print(f"  4. 시간별 안정성이 낮은 피처는 주의")
print("=" * 70)